In [42]:
import requests
from typing import Dict, Optional

class Usuario:
    def __init__(self, nome: str, senha: str):
        self.nome = nome
        self.senha = senha
        # Estrutura: {id: {"dados": dict, "qtd": int}}
        self.colecao: Dict[int, dict] = {}
        self.favorito_id: Optional[int] = None

class Pokemon:
    _base_url = "https://pokeapi.co/api/v2/pokemon/"

    def __init__(self, nome: str, tipos: list, altura: float, peso: float, p_id: Optional[int] = None, moves: Optional[list] = None, stats: Optional[list] = None):
        self.nome = nome
        self.tipos = tipos
        self.altura = altura
        self.peso = peso
        self.id = p_id
        self.moves = moves if moves is not None else []
        self.stats = stats if stats is not None else []

    @classmethod
    def from_api(cls, pokemon_identifier: str | int):
        try:
            response = requests.get(f"{cls._base_url}{str(pokemon_identifier).lower().strip()}")
            response.raise_for_status()
            data = response.json()

            return cls(
                nome=data["name"].capitalize(),
                tipos=[t["type"]["name"] for t in data["types"]],
                altura=data["height"] / 10,
                peso=data["weight"] / 10,
                p_id=data["id"],
                moves=[move["move"]["name"] for move in data["moves"]],
                stats=data["stats"]
            )
        except requests.exceptions.HTTPError:
            print(f"ERRO - Pokémon '{pokemon_identifier}' inexistente.")
            return None
        except Exception as e:
            print(f"ERRO - Falha na requisição para '{pokemon_identifier}': {e}")
            return None

class PokeSystem:
    def __init__(self):
        self.usuarios: Dict[str, Usuario] = {}

    def criar_conta(self, nome: str, senha: str):
        if nome in self.usuarios:
            print(f"ERRO - Usuário já existe: {nome}")
            return
        self.usuarios[nome] = Usuario(nome, senha)
        print(f"OK - Usuário '{nome}' criado com sucesso!")

    def excluir_usuario(self, nome: str):
        if nome not in self.usuarios:
            print(f"ERRO - Usuário não encontrado: {nome}")
            return
        del self.usuarios[nome]
        print(f"OK - Usuário '{nome}' excluído do sistema.")

    def adicionar_pokemon(self, nome_usuario: str, busca: str):
        if nome_usuario not in self.usuarios:
            print(f"ERRO - Usuário não encontrado: {nome_usuario}")
            return

        try:
            pokemon_obj = Pokemon.from_api(busca)
            if pokemon_obj is None:
                return

            p_id = pokemon_obj.id
            user = self.usuarios[nome_usuario]

            # Lógica de contagem (Evita duplicata exata no retorno, apenas incrementa)
            if p_id in user.colecao:
                user.colecao[p_id]["qtd"] += 1
            else:
                user.colecao[p_id] = {
                    "qtd": 1,
                    "dados": {
                        "nome": pokemon_obj.nome,
                        "tipos": pokemon_obj.tipos,
                        "altura": pokemon_obj.altura,
                        "peso": pokemon_obj.peso
                    }
                }
            print(f"OK - {pokemon_obj.nome} atribuído a {nome_usuario}")

        except Exception as e:
            print(f"ERRO - Falha na operação ao adicionar pokemon: {e}")

    def consultar_pokemons(self, pokemon: str):
        try:
            pokemon_obj = Pokemon.from_api(pokemon)
            if pokemon_obj is None:
                return
            print(f"\n====================================================")
            print(f"INFO POKEMON: {pokemon_obj.nome}")
            print(f"ID: {pokemon_obj.id}")
            print(f"Tipos: {', '.join(pokemon_obj.tipos)}")
            print(f"====================================================")
            print(f"Altura: {pokemon_obj.altura}m")
            print(f"Peso: {pokemon_obj.peso}kg")
            print(f"====================================================")
            print(f"Status base:")
            for stat in pokemon_obj.stats:
                print(f"{stat['stat']['name']}: {stat['base_stat']}")
            print(f"====================================================")
            print("Golpes que aprende: \n " + "\n ".join(pokemon_obj.moves).upper())
            print(f"====================================================")
            return
        except Exception as e:
            print(f"ERRO - Falha na consulta de pokemons: {e}")
            return None

    def definir_favorito(self, nome_usuario: str, p_id: int):
        user = self.usuarios.get(nome_usuario)
        if not user:
            print("ERRO - Usuário não encontrado.")
            return

        if user.favorito_id is not None:
            print(f"ERRO - Usuário '{nome_usuario}' já possui um favorito (ID: {user.favorito_id})")
            return

        pokemon_name = None
        if p_id in user.colecao:
            pokemon_name = user.colecao[p_id]['dados']['nome']
        else:
            pokemon_obj = Pokemon.from_api(p_id)
            if pokemon_obj is None:
                return
            pokemon_name = pokemon_obj.nome

        user.favorito_id = p_id
        print(f"OK - {pokemon_name} definido como favorito!")

    def remover_favorito(self, nome_usuario: str):
        user = self.usuarios.get(nome_usuario)
        if not user:
            print("ERRO - Usuário não encontrado.")
            return

        if user.favorito_id is None:
            print(f"ERRO - Usuário '{nome_usuario}' não possui um favorito.")
            return

        user.favorito_id = None
        print(f"OK - Favorito de {nome_usuario} removido.")


    def remover_pokemon(self, nome_usuario: str, p_id: int):
        user = self.usuarios.get(nome_usuario)
        if not user:
            print("ERRO - Usuário não encontrado.")
            return

        if p_id not in user.colecao:
            print(f"ERRO - Pokémon ID {p_id} não está na coleção de {nome_usuario}")
            return

        pokemon_name_to_remove = user.colecao[p_id]['dados']['nome']

        if user.favorito_id == p_id:
           print(f"Aviso, você está prestes a excluir de sua coleção o seu pokemon favorito ({pokemon_name_to_remove}). Deseja continuar?")
           resposta = str(input("Pressione S para sim e N para não: "))
           if resposta.lower() == "s":
              del user.colecao[p_id]
              print(f"OK - {pokemon_name_to_remove} removido da coleção de {nome_usuario}")
              if user.favorito_id == p_id:
                  user.favorito_id = None
              return
           elif resposta.lower() == "n":
              print("Operação cancelada")
              return
           else:
              print("Resposta inválida")
              return
        else:
          del user.colecao[p_id]
          print(f"OK - {pokemon_name_to_remove} removido da coleção de {nome_usuario}")



    def consultar_informacoes(self, nome_usuario: str):
        user = self.usuarios.get(nome_usuario)
        if not user:
            print("ERRO - Usuário não encontrado.")
            return

        print(f"\nINFO USUÁRIO: {user.nome.upper()}")

        # Display favorite first, if any
        if user.favorito_id is not None:
            fav_id = user.favorito_id
            fav_pokemon_info_str = ""

            if fav_id in user.colecao:
                d = user.colecao[fav_id]['dados']
                qtd = user.colecao[fav_id]['qtd']
                fav_pokemon_info_str = (
                    f"{d['nome']} (ID: {fav_id}) | Tipos: {', '.join(d['tipos'])} | "
                    f"{d['altura']}m - {d['peso']}kg | ({qtd}) ⭐ [FAVORITO]"
                )
            else:
                pokemon_obj = Pokemon.from_api(fav_id)
                if pokemon_obj:
                    fav_pokemon_info_str = (
                        f"{pokemon_obj.nome} (ID: {pokemon_obj.id}) | Tipos: {', '.join(pokemon_obj.tipos)} | "
                        f"{pokemon_obj.altura}m - {pokemon_obj.peso}kg | (Não presente na coleção) ⭐ [FAVORITO]"
                    )
                else:
                    print(f"ATENÇÃO: Pokémon favorito (ID: {fav_id}) não encontrado na API ou na coleção.")

            if fav_pokemon_info_str:
                print(fav_pokemon_info_str)
        else:
            print("Nenhum favorito definido.")


        if not user.colecao:
            print("Coleção vazia.")
            return

        print("\nCOLEÇÃO:")
        for p_id, item in user.colecao.items():
            if user.favorito_id == p_id:
               print(f"{d['nome']} (ID: {p_id}) | Tipos: {', '.join(d['tipos'])} | {d['altura']}m - {d['peso']}kg | ({item['qtd']}) ⭐ [FAVORITO]")
               continue

            d = item["dados"]
            print(f"{d['nome']} (ID: {p_id}) | Tipos: {', '.join(d['tipos'])} | {d['altura']}m - {d['peso']}kg | ({item['qtd']})")

    def login_teste(self, nome: str, senha_digitada: str):
        #Apenas para validação de cenário de teste
        user = self.usuarios.get(nome)
        if not user or user.senha != senha_digitada:
            print(f"ERRO - Falha no login para '{nome}': Senha incorreta ou usuário inexistente.")
        else:
            print(f"OK - Login bem-sucedido para '{nome}'.")

# Instanciação do sistema
# [apelido do sistema] = PokeSystem()

In [7]:
poke_api = PokeSystem()

# 1. Testar Cadastro e Duplicidade de Usuário
poke_api.criar_conta('ash', '123')
poke_api.criar_conta('ash', '456') # Deve retornar ERRO

# 2. Testar Login com Senha Incorreta
poke_api.login_teste('ash', 'senha_errada') # Deve retornar ERRO

# 3. Vincular Pokémon Inexistente
poke_api.adicionar_pokemon('ash', 'agumon') # Deve retornar ERRO

# 4. Adicionar Pokémons e Testar Contagem (Duplicados)
poke_api.adicionar_pokemon('ash', 'articuno')
poke_api.adicionar_pokemon('ash', '144') # ID do Articuno para testar contagem
poke_api.adicionar_pokemon('ash', 'pikachu')

# 5. Atribuir Favoritos (Testar rejeição de mais de 1)
poke_api.definir_favorito('ash', 144) # Articuno favorito
poke_api.definir_favorito('ash', 25)  # Pikachu (Deve retornar ERRO - Já possui favorito)

# 6. Consulta de Informações
poke_api.consultar_informacoes('ash')

# 7. Excluir Usuário
poke_api.excluir_usuario('ash')
poke_api.consultar_informacoes('ash') # Deve retornar ERRO - Não encontrado

OK - Usuário 'ash' criado com sucesso!
ERRO - Usuário já existe: ash
ERRO - Falha no login para 'ash': Senha incorreta ou usuário inexistente.
ERRO - Pokémon 'agumon' inexistente.
OK - Articuno atribuído a ash
OK - Articuno atribuído a ash
OK - Pikachu atribuído a ash
OK - Articuno definido como favorito!
ERRO - Usuário 'ash' já possui um favorito (ID: 144)

INFO USUÁRIO: ASH
Articuno (ID: 144) | Tipos: ice, flying | 1.7m - 55.4kg | (2) ⭐ [FAVORITO]
Pikachu (ID: 25) | Tipos: electric | 0.4m - 6.0kg | (1)
OK - Usuário 'ash' excluído do sistema.
ERRO - Usuário não encontrado.


In [43]:
poke_api = PokeSystem()

#8. Testar Consultar Pokémon favorito fora da coleção
poke_api.criar_conta('ash', '123')
poke_api.definir_favorito('ash', 150)
poke_api.consultar_informacoes('ash')

#9. Testar Remover favorito
poke_api.remover_favorito('ash')

#10. Testar Exibição de Pokemon Favorito
poke_api.adicionar_pokemon('ash', 150)
poke_api.definir_favorito('ash', 150)
poke_api.consultar_informacoes('ash')

#11. Testar Consulta de Pokémon
poke_api.consultar_pokemons('hitmontop')

OK - Usuário 'ash' criado com sucesso!
OK - Mewtwo definido como favorito!

INFO USUÁRIO: ASH
Mewtwo (ID: 150) | Tipos: psychic | 2.0m - 122.0kg | (Não presente na coleção) ⭐ [FAVORITO]
Coleção vazia.
OK - Favorito de ash removido.
OK - Mewtwo atribuído a ash
OK - Mewtwo definido como favorito!

INFO USUÁRIO: ASH
Mewtwo (ID: 150) | Tipos: psychic | 2.0m - 122.0kg | (1) ⭐ [FAVORITO]

COLEÇÃO:
Mewtwo (ID: 150) | Tipos: psychic | 2.0m - 122.0kg | (1) ⭐ [FAVORITO]

INFO POKEMON: Hitmontop
ID: 237
Tipos: fighting
Altura: 1.4m
Peso: 48.0kg
Status base:
hp: 50
attack: 95
defense: 95
special-attack: 35
special-defense: 110
speed: 70
Golpes que aprende: 
 MEGA-PUNCH
 MEGA-KICK
 ROLLING-KICK
 HEADBUTT
 TACKLE
 BODY-SLAM
 TAKE-DOWN
 DOUBLE-EDGE
 LOW-KICK
 COUNTER
 SEISMIC-TOSS
 STRENGTH
 EARTHQUAKE
 DIG
 TOXIC
 AGILITY
 QUICK-ATTACK
 MIMIC
 DOUBLE-TEAM
 FOCUS-ENERGY
 SWIFT
 HIGH-JUMP-KICK
 REST
 ROCK-SLIDE
 SUBSTITUTE
 TRIPLE-KICK
 THIEF
 SNORE
 CURSE
 REVERSAL
 PROTECT
 MACH-PUNCH
 MUD-SLAP
 DET